# 行业配置策略：资金流向视角
## 基于华泰证券研报复现

本notebook实现以下功能：
1. 数据获取：北向资金、两融资金、ETF资金、产业资本数据
2. 资金流向指标构建
3. 行业轮动策略回测
4. 复合指标与策略叠加
5. 结果可视化

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from config import settings
from source import DataLoader, NorthboundFunds, MarginFunds, ETFFunds
from source import IndustrialCapital, IndicatorCalculator
from source import IndustryRotationStrategy, CompositeIndicator

## 1. 初始化数据加载器

In [ ]:
dl = DataLoader()
ic = IndicatorCalculator(dl)
north = NorthboundFunds(dl)
margin = MarginFunds(dl)
etf = ETFFunds(dl)
ic_capital = IndustrialCapital(dl)
strategy = IndustryRotationStrategy(dl, ic)
composite = CompositeIndicator(dl, ic)

print("数据加载器初始化完成")

## 2. 获取北向资金数据

In [ ]:
print("正在获取北向资金数据...")
north_df = north.get_northbound_net_inflow()
print(f"北向资金数据形状: {north_df.shape}")
north_df.head()

In [ ]:
north_indicators = north.build_north_indicators(freq='W')
print(f"北向资金指标数量: {len(north_indicators)}")
for ind in north_indicators:
    if len(ind) > 0:
        print(f"  - {ind['indicator_name'].iloc[0]}: {len(ind)} 行")

## 3. 获取两融资金数据

In [ ]:
print("正在获取两融资金数据...")
margin_df = margin.get_margin_summary()
print(f"两融资金数据形状: {margin_df.shape}")
margin_df.head()

In [ ]:
margin_indicators = margin.build_margin_indicators(freq='W')
print(f"两融资金指标数量: {len(margin_indicators)}")
for ind in margin_indicators:
    if len(ind) > 0:
        print(f"  - {ind['indicator_name'].iloc[0]}: {len(ind)} 行")

## 4. 获取ETF资金数据

In [ ]:
print("正在获取ETF列表...")
etf_list = etf.get_etf_list()
print(f"ETF数量: {len(etf_list)}")
etf_list.head()

In [ ]:
print("正在获取行业ETF资金流向...")
sector_etfs = etf.get_sector_etf_list()
print(f"行业主题ETF数量: {len(sector_etfs)}")

## 5. 获取产业资本数据

In [ ]:
print("正在获取产业资本数据...")
seo_indicators = ic_capital.build_seo_indicators(freq='W')
print(f"定向增发指标数量: {len(seo_indicators)}")

float_indicators = ic_capital.build_float_indicators(freq='W')
print(f"限售解禁指标数量: {len(float_indicators)}")

repurchase_indicators = ic_capital.build_repurchase_indicators(freq='W')
print(f"回购指标数量: {len(repurchase_indicators)}")

## 6. 获取中信行业列表

In [ ]:
print("正在获取中信行业列表...")
industry_list = ic.get_industry_list(level='L1')
print(f"一级行业数量: {len(industry_list)}")
print(industry_list)

In [ ]:
print("正在获取行业日线数据...")
industry_returns = strategy.get_industry_returns(industry_list['code'].tolist())
print(f"行业收益数据形状: {industry_returns.shape}")

## 7. 北向资金策略回测

In [ ]:
print("正在运行北向资金策略回测...")
if len(north_indicators) > 0:
    north_ind = north_indicators[0]
    
    merged = north_ind.merge(
        industry_returns,
        on='period',
        how='inner'
    )
    
    if len(merged) > 0:
        strat_results = strategy.run_stratification_test(
            merged.dropna(subset=['net_amount']),
            'net_amount'
        )
        print("分层回测结果:")
        print(strat_results)
        
        plt.figure(figsize=(10, 6))
        plt.bar(strat_results['group'], strat_results['mean_return'])
        plt.xlabel('Group')
        plt.ylabel('Mean Return')
        plt.title('Northbound Funds Strategy - Stratification Results')
        plt.show()

## 8. 两融资金策略回测

In [ ]:
print("正在运行两融资金策略回测...")
if len(margin_indicators) > 0:
    margin_ind = margin_indicators[0]
    
    merged = margin_ind.merge(
        industry_returns,
        on='period',
        how='inner'
    )
    
    if len(merged) > 0:
        strat_results = strategy.run_stratification_test(
            merged.dropna(subset=['balance']),
            'balance'
        )
        print("分层回测结果:")
        print(strat_results)

## 9. 复合资金流向指标

In [ ]:
print("正在构建复合资金流向指标...")

combined_df = composite.combine_indicators(
    [ind for ind in north_indicators + margin_indicators if len(ind) > 0],
    method='equal_weight'
)
print(f"复合指标数据形状: {combined_df.shape if len(combined_df) > 0 else '无数据'}")

## 10. 策略评估与可视化

In [ ]:
print("正在评估策略表现...")

benchmark_returns = strategy.get_benchmark_returns()
print(f"基准收益数据形状: {benchmark_returns.shape if len(benchmark_returns) > 0 else '无数据'}")

In [ ]:
print("\n=== 策略评估结果 ===\n")

print("注意: 本项目需要完整的历史数据才能运行完整的回测。")
print("部分数据接口可能需要 Wind 或其他付费数据源支持。")
print("\n研报中提到的核心发现:")
print("1. 北向资金用于行业轮动策略效果最好")
print("2. 两融和趋势型ETF资金流指标居次")
print("3. 定向增发现象属于正向配置机会")
print("4. 限售解禁和大股东减持会拖累行业表现")
print("5. 综合指标多头年化超额收益超过10%")

## 11. 数据缺失说明

In [ ]:
print("=" * 60)
print("数据缺失说明")
print("=" * 60)
print("""
研报复现过程中可能存在以下数据缺失:

1. 北向资金行业归属数据
   - tushare的北向资金数据主要是持股明细
   - 需要额外处理才能归因到中信行业

2. 两融资金行业明细
   - 两融数据主要是全市场汇总
   - 行业层面的两融数据需要通过成分股计算

3. ETF资金流向行业归因
   - 需要ETF持仓明细才能精确归因
   - 当前主要获取全市场ETF份额变化

4. 产业资本详细数据
   - 定向增发、限售解禁、回购等
   - 需要完整的个股事件数据

5. 行业景气度数据
   - 用于与资金流向策略叠加
   - 需要财务数据和一致预期数据

建议补充数据源:
- Wind终端数据
- Choice金融终端
- 聚源数据库
""")

## 12. 核心函数调用示例

In [ ]:
print("核心函数调用示例:\n")

print("1. 获取北向资金日度净流入:")
print("   north_df = north.get_northbound_net_inflow()")

print("\n2. 构建北向资金指标:")
print("   north_indicators = north.build_north_indicators(freq='W')")

print("\n3. 获取两融资金汇总:")
print("   margin_df = margin.get_margin_summary()")

print("\n4. 构建两融资金指标:")
print("   margin_indicators = margin.build_margin_indicators(freq='W')")

print("\n5. 获取行业列表:")
print("   industry_list = ic.get_industry_list(level='L1')")

print("\n6. 获取行业收益:")
print("   industry_returns = strategy.get_industry_returns(code_list)")

print("\n7. 运行分层回测:")
print("   strat_results = strategy.run_stratification_test(df, indicator_col)")

print("\n8. 构建复合指标:")
print("   combined = composite.combine_indicators(indicator_list)")